# Class-imbalance-aware Deep Learning for Skin Cancer Detection from Lesion Images

**Single-file Google Colab notebook** for a conference-paper style binary skin lesion classification experiment: **benign vs malignant**.

This notebook automatically downloads a Kaggle-hosted ISIC resized dataset, prepares stratified splits, trains EfficientNet and ConvNeXt models with multiple imbalance-aware strategies, evaluates each experiment, saves paper-ready figures/tables, and optionally generates Grad-CAM visualizations.

> Default dataset: `nischaydnk/isic-2019-jpg-224x224-resized`, which contains `train-image/`, `train-metadata.csv`, and a binary `target` column where `0 = benign`, `1 = malignant`.

No experimental results are hard-coded. All results are computed when you run the notebook.

## 1. Install dependencies

Run this cell first. It installs only packages that are usually missing from Colab.

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA RTX A5000


In [2]:
import sys
print(sys.executable)

/home/22010759/miniconda3/envs/ml/bin/python


In [3]:
import sys
import subprocess
import importlib.util

## 2. Imports, configuration, and reproducibility

For a quick debugging run, set:

```python
CONFIG["run"]["experiment_plan"] = "quick"
CONFIG["dataset"]["max_samples"] = 1000
CONFIG["training"]["epochs"] = 1
```

For paper experiments, use the default `core` plan or switch to `full`.

In [4]:
import os
import json
import math
import random
import shutil
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

import torchvision.transforms as T
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

import matplotlib.pyplot as plt
import cv2

CONFIG = {
    "project_name": "skin-cancer-class-imbalance",
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "dataset": {
        "auto_download": True,
        "source": "kagglehub",
        "kaggle_slug": "nischaydnk/isic-2019-jpg-224x224-resized",
        "manual_root": None,
        "csv_path": None,
        "image_dir": None,
        "target_col": "target",
        "image_id_col": None,
        "max_samples": None,
        "test_size": 0.15,
        "val_size": 0.15,
    },
    "training": {
        "image_size": 224,
        "batch_size": 32,
        "num_workers": 2,
        "epochs": 20,
        "learning_rate": 3e-4,
        "weight_decay": 1e-4,
        "early_stopping_patience": 3,
        "threshold": 0.5,
        "use_amp": True,
    },
    "augmentation": {
        "minority_stronger_aug": False,
        "rotation_degrees": 25,
        "color_jitter": {
            "brightness": 0.15,
            "contrast": 0.15,
            "saturation": 0.10,
            "hue": 0.02,
        },
    },
    "loss": {
        "focal_alpha": 0.25,
        "focal_gamma": 2.0,
        "class_balanced_beta": 0.9999,
    },
    "run": {
        "experiment_plan": "core",  # options: quick, core, full
        "save_all_checkpoints": False,
        "make_gradcam": True,
        "continue_on_experiment_error": True,
    },
    "outputs": {
        "root": "content/skin_cancer_outputs",
    },
}

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(CONFIG["seed"])
DEVICE = torch.device(CONFIG["device"])
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

OUTPUT_ROOT = Path(CONFIG["outputs"]["root"])
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
LOG_DIR = OUTPUT_ROOT / "logs"
FIGURE_DIR = OUTPUT_ROOT / "figures"
RESULT_DIR = OUTPUT_ROOT / "results"
for d in [CHECKPOINT_DIR, LOG_DIR, FIGURE_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_ROOT / "config_used.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
print("Outputs will be saved to:", OUTPUT_ROOT)

/home/22010759/miniconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA RTX A5000
Outputs will be saved to: content/skin_cancer_outputs


## 3. Automatic dataset download

The default download uses `kagglehub.dataset_download`. Public Kaggle datasets often download directly. If Kaggle asks for authentication, add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Colab secrets, or upload `kaggle.json` using the fallback code in this cell.

In [5]:
def discover_dataset_files(root: Path) -> Tuple[Path, Path]:
    """Find metadata CSV and image directory automatically."""
    csv_candidates = list(root.rglob("*.csv"))
    if not csv_candidates:
        raise FileNotFoundError(f"No CSV file found under {root}")

    preferred_names = ["train-metadata.csv", "metadata.csv", "train.csv"]
    csv_path = None
    for name in preferred_names:
        matches = [p for p in csv_candidates if p.name.lower() == name]
        if matches:
            csv_path = matches[0]
            break
    if csv_path is None:
        csv_path = csv_candidates[0]

    image_exts = {".jpg", ".jpeg", ".png"}
    image_files = [p for p in root.rglob("*") if p.suffix.lower() in image_exts]
    if not image_files:
        raise FileNotFoundError(f"No image files found under {root}")

    # Prefer a directory called train-image or images if present.
    image_dirs = sorted({p.parent for p in image_files}, key=lambda x: len(str(x)))
    preferred_dir = None
    for dir_name in ["train-image", "train_images", "images", "image"]:
        matches = [d for d in image_dirs if d.name.lower() == dir_name]
        if matches:
            preferred_dir = matches[0]
            break
    if preferred_dir is None:
        # Pick the directory containing the most images.
        counts = {}
        for p in image_files:
            counts[p.parent] = counts.get(p.parent, 0) + 1
        preferred_dir = max(counts, key=counts.get)

    print("Metadata CSV:", csv_path)
    print("Image directory:", preferred_dir)
    return csv_path, preferred_dir

DATASET_ROOT = Path("./content/dataset")
if CONFIG["dataset"].get("csv_path") and CONFIG["dataset"].get("image_dir"):
    CSV_PATH = Path(CONFIG["dataset"]["csv_path"])
    IMAGE_DIR = Path(CONFIG["dataset"]["image_dir"])
else:
    CSV_PATH, IMAGE_DIR = discover_dataset_files(DATASET_ROOT)

print("CSV_PATH =", CSV_PATH)
print("IMAGE_DIR =", IMAGE_DIR)

Metadata CSV: content/dataset/train-metadata.csv
Image directory: content/dataset/image
CSV_PATH = content/dataset/train-metadata.csv
IMAGE_DIR = content/dataset/image


## 4. Dataset preparation and stratified split

This cell resolves image paths, cleans labels, reports class imbalance, and creates stratified train/validation/test splits.

In [6]:
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]


def infer_target_column(df: pd.DataFrame, preferred: Optional[str] = None) -> str:
    if preferred and preferred in df.columns:
        return preferred
    for col in ["target", "label", "class", "benign_malignant", "diagnosis"]:
        if col in df.columns:
            return col
    raise ValueError(f"Could not infer target column. Columns: {list(df.columns)}")


def convert_binary_target(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(float).astype("Int64")
    mapping = {
        "benign": 0,
        "malignant": 1,
        "0": 0,
        "1": 1,
        "false": 0,
        "true": 1,
        "nevus": 0,
        "melanoma": 1,
        "basal cell carcinoma": 1,
        "squamous cell carcinoma": 1,
    }
    return series.astype(str).str.strip().str.lower().map(mapping).astype("Int64")


def build_image_index(image_dir: Path) -> Dict[str, str]:
    image_paths = []
    for ext in IMAGE_EXTENSIONS:
        image_paths.extend(image_dir.rglob(f"*{ext}"))
        image_paths.extend(image_dir.rglob(f"*{ext.upper()}"))
    image_index = {}
    for p in image_paths:
        image_index[p.name] = str(p)
        image_index[p.stem] = str(p)
    print(f"Indexed {len(image_paths):,} image files from {image_dir}")
    return image_index


def infer_image_id_columns(df: pd.DataFrame, preferred: Optional[str] = None) -> List[str]:
    if preferred and preferred in df.columns:
        return [preferred]
    candidates = [
        "image", "image_name", "image_id", "isic_id", "filename", "file_name", "path", "filepath"
    ]
    found = [c for c in candidates if c in df.columns]
    if found:
        return found
    # fallback: try object columns
    return [c for c in df.columns if df[c].dtype == "object"]


def resolve_image_paths(df: pd.DataFrame, image_dir: Path, image_id_col: Optional[str] = None) -> pd.DataFrame:
    image_index = build_image_index(image_dir)
    candidate_cols = infer_image_id_columns(df, image_id_col)
    print("Candidate image id columns:", candidate_cols)

    resolved_paths = []
    resolved_from_col = []
    for _, row in df.iterrows():
        found_path = None
        found_col = None
        for col in candidate_cols:
            val = str(row[col]).strip()
            if val in image_index:
                found_path = image_index[val]
                found_col = col
                break
            stem = Path(val).stem
            if stem in image_index:
                found_path = image_index[stem]
                found_col = col
                break
            for ext in IMAGE_EXTENSIONS:
                candidate = val + ext
                if candidate in image_index:
                    found_path = image_index[candidate]
                    found_col = col
                    break
            if found_path:
                break
        resolved_paths.append(found_path)
        resolved_from_col.append(found_col)

    out = df.copy()
    out["image_path"] = resolved_paths
    out["image_path_source_col"] = resolved_from_col
    missing = out["image_path"].isna().sum()
    if missing > 0:
        print(f"Warning: dropping {missing:,} rows with unresolved image paths.")
        out = out.dropna(subset=["image_path"]).reset_index(drop=True)
    print("Resolved dataframe shape:", out.shape)
    return out


def load_and_prepare_metadata(config: Dict, csv_path: Path, image_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    print("Raw metadata shape:", df.shape)
    print("Columns:", list(df.columns))

    target_col = infer_target_column(df, config["dataset"].get("target_col"))
    df["target"] = convert_binary_target(df[target_col])
    before = len(df)
    df = df.dropna(subset=["target"]).copy()
    df["target"] = df["target"].astype(int)
    df = df[df["target"].isin([0, 1])].copy()
    print(f"Dropped {before - len(df):,} rows with invalid/missing target labels.")

    df = resolve_image_paths(df, image_dir, config["dataset"].get("image_id_col"))

    max_samples = config["dataset"].get("max_samples")
    if max_samples is not None and max_samples < len(df):
        df, _ = train_test_split(
            df,
            train_size=max_samples,
            stratify=df["target"],
            random_state=config["seed"],
        )
        df = df.reset_index(drop=True)
        print(f"Subsampled to {len(df):,} images for debugging.")

    counts = df["target"].value_counts().sort_index()
    print("\nClass distribution:")
    print(counts.rename(index={0: "benign_0", 1: "malignant_1"}))
    if 0 in counts.index and 1 in counts.index:
        print(f"Imbalance ratio benign:malignant = {counts.loc[0] / max(counts.loc[1], 1):.2f}:1")
    return df.reset_index(drop=True)


def make_stratified_splits(df: pd.DataFrame, config: Dict) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    test_size = config["dataset"]["test_size"]
    val_size = config["dataset"]["val_size"]
    seed = config["seed"]

    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["target"],
        random_state=seed,
    )
    relative_val_size = val_size / (1.0 - test_size)
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=relative_val_size,
        stratify=train_val_df["target"],
        random_state=seed,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        counts = split_df["target"].value_counts().sort_index().to_dict()
        print(f"{name:>5}: n={len(split_df):,}, class_counts={counts}")
        split_df.to_csv(RESULT_DIR / f"{name}_split.csv", index=False)
    return train_df, val_df, test_df

metadata_df = load_and_prepare_metadata(CONFIG, CSV_PATH, IMAGE_DIR)
train_df, val_df, test_df = make_stratified_splits(metadata_df, CONFIG)

Raw metadata shape: (25331, 4)
Columns: ['Unnamed: 0', 'isic_id', 'patient_id', 'target']
Dropped 0 rows with invalid/missing target labels.
Indexed 25,331 image files from content/dataset/image
Candidate image id columns: ['isic_id']
Resolved dataframe shape: (25331, 6)

Class distribution:
target
benign_0       20809
malignant_1     4522
Name: count, dtype: int64
Imbalance ratio benign:malignant = 4.60:1
train: n=17,731, class_counts={0: 14565, 1: 3166}
  val: n=3,800, class_counts={0: 3122, 1: 678}
 test: n=3,800, class_counts={0: 3122, 1: 678}


## 5. Dataset class and augmentations

The training transform includes resize, flips, rotation, color jitter, tensor conversion, and ImageNet normalization.

In [7]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_transforms(split: str, config: Dict, stronger: bool = False):
    image_size = config["training"]["image_size"]
    aug_cfg = config["augmentation"]
    jitter = aug_cfg["color_jitter"]

    if split == "train":
        rotation = aug_cfg["rotation_degrees"] * (1.5 if stronger else 1.0)
        brightness = jitter["brightness"] * (1.5 if stronger else 1.0)
        contrast = jitter["contrast"] * (1.5 if stronger else 1.0)
        saturation = jitter["saturation"] * (1.5 if stronger else 1.0)
        hue = min(jitter["hue"] * (1.5 if stronger else 1.0), 0.08)
        return T.Compose([
            T.Resize((image_size, image_size)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.RandomRotation(degrees=rotation),
            T.ColorJitter(brightness=brightness, contrast=contrast, saturation=saturation, hue=hue),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    return T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


class SkinLesionDataset(Dataset):
    """Binary skin lesion dataset reading image paths and target labels from a dataframe."""

    def __init__(self, df: pd.DataFrame, transform=None, minority_transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.minority_transform = minority_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row["image_path"]
        target = float(row["target"])
        image = Image.open(image_path).convert("RGB")

        if self.minority_transform is not None and int(target) == 1:
            image = self.minority_transform(image)
        elif self.transform is not None:
            image = self.transform(image)

        target_tensor = torch.tensor(target, dtype=torch.float32)
        return {
            "image": image,
            "target": target_tensor,
            "image_path": image_path,
        }


def make_weighted_sampler(df: pd.DataFrame) -> WeightedRandomSampler:
    targets = df["target"].values.astype(int)
    class_counts = np.bincount(targets, minlength=2)
    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_weights = class_weights[targets]
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )
    return sampler


def make_dataloaders(train_df, val_df, test_df, config: Dict, use_weighted_sampler: bool = False):
    train_transform = get_transforms("train", config)
    minority_transform = None
    if config["augmentation"].get("minority_stronger_aug", False):
        minority_transform = get_transforms("train", config, stronger=True)

    val_transform = get_transforms("val", config)
    train_ds = SkinLesionDataset(train_df, transform=train_transform, minority_transform=minority_transform)
    val_ds = SkinLesionDataset(val_df, transform=val_transform)
    test_ds = SkinLesionDataset(test_df, transform=val_transform)

    sampler = make_weighted_sampler(train_df) if use_weighted_sampler else None
    shuffle = sampler is None

    loader_kwargs = dict(
        batch_size=config["training"]["batch_size"],
        num_workers=config["training"]["num_workers"],
        pin_memory=torch.cuda.is_available(),
    )
    train_loader = DataLoader(train_ds, shuffle=shuffle, sampler=sampler, **loader_kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)
    return train_loader, val_loader, test_loader

print("Dataset and transform code ready.")

Dataset and transform code ready.


## 6. Models: EfficientNet and ConvNeXt

All models output **one binary logit**. The sigmoid is applied only during metric computation/evaluation.

In [8]:
MODEL_NAME_MAP = {
    "efficientnet_b0": "efficientnet_b0",
    "efficientnet_b3": "efficientnet_b3",
    "convnext_tiny": "convnext_tiny",
}


def build_model(model_key: str, pretrained: bool = True) -> nn.Module:
    if model_key not in MODEL_NAME_MAP:
        raise ValueError(f"Unknown model_key={model_key}. Available: {list(MODEL_NAME_MAP)}")
    model_name = MODEL_NAME_MAP[model_key]
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=1)
    return model


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for key in MODEL_NAME_MAP:
    print(key, "->", MODEL_NAME_MAP[key])

efficientnet_b0 -> efficientnet_b0
efficientnet_b3 -> efficientnet_b3
convnext_tiny -> convnext_tiny


## 7. Loss functions for class imbalance

Implemented losses:

- BCE with logits
- Weighted BCE with `pos_weight = num_negative / num_positive`
- Focal Loss
- Class-Balanced Focal Loss using effective number of samples

In [9]:
class FocalLoss(nn.Module):
    """Binary focal loss over logits."""

    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        logits = logits.view(-1)
        targets = targets.view(-1)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(
            targets == 1,
            torch.full_like(targets, self.alpha),
            torch.full_like(targets, 1.0 - self.alpha),
        )
        loss = alpha_t * (1 - pt).pow(self.gamma) * bce
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


class ClassBalancedFocalLoss(nn.Module):
    """Class-balanced focal loss using effective number of samples."""

    def __init__(self, class_counts: List[int], beta: float = 0.9999, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        if len(class_counts) != 2:
            raise ValueError("Binary classification requires class_counts=[num_negative, num_positive].")
        counts = np.array(class_counts, dtype=np.float64)
        effective_num = 1.0 - np.power(beta, counts)
        weights = (1.0 - beta) / np.maximum(effective_num, 1e-12)
        weights = weights / weights.sum() * 2.0
        self.register_buffer("class_weights", torch.tensor(weights, dtype=torch.float32))
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        logits = logits.view(-1)
        targets = targets.view(-1)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        target_long = targets.long()
        alpha_t = self.class_weights[target_long]
        loss = alpha_t * (1 - pt).pow(self.gamma) * bce
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


def build_loss(loss_name: str, train_df: pd.DataFrame, config: Dict) -> nn.Module:
    targets = train_df["target"].values.astype(int)
    class_counts = np.bincount(targets, minlength=2)
    num_neg, num_pos = int(class_counts[0]), int(class_counts[1])

    if loss_name == "bce":
        return nn.BCEWithLogitsLoss()

    if loss_name == "weighted_bce":
        pos_weight_value = num_neg / max(num_pos, 1)
        pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)
        print(f"Using weighted BCE pos_weight={pos_weight_value:.4f}")
        return nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    if loss_name == "focal":
        return FocalLoss(
            alpha=config["loss"]["focal_alpha"],
            gamma=config["loss"]["focal_gamma"],
        )

    if loss_name == "class_balanced_focal":
        return ClassBalancedFocalLoss(
            class_counts=[num_neg, num_pos],
            beta=config["loss"]["class_balanced_beta"],
            gamma=config["loss"]["focal_gamma"],
        ).to(DEVICE)

    raise ValueError(f"Unknown loss_name={loss_name}")

print("Loss functions ready.")

Loss functions ready.


## 8. Metrics and plotting utilities

These functions save paper-ready ROC curves, confusion matrices, and training curves.

In [10]:
def compute_metrics(y_true, y_prob, threshold: float = 0.5) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {}
    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
        metrics["auprc"] = average_precision_score(y_true, y_prob)
    else:
        metrics["roc_auc"] = np.nan
        metrics["auprc"] = np.nan

    metrics["accuracy"] = accuracy_score(y_true, y_pred)
    metrics["precision"] = precision_score(y_true, y_pred, zero_division=0)
    metrics["recall_sensitivity"] = recall_score(y_true, y_pred, zero_division=0)
    metrics["f1"] = f1_score(y_true, y_pred, zero_division=0)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    metrics["specificity"] = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    metrics["tn"] = int(tn)
    metrics["fp"] = int(fp)
    metrics["fn"] = int(fn)
    metrics["tp"] = int(tp)
    return metrics


def plot_roc_curve(y_true, y_prob, title: str, save_path: Path) -> None:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    if len(np.unique(y_true)) < 2:
        print("Skipping ROC curve because only one class is present.")
        return
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_value = roc_auc_score(y_true, y_prob)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC = {auc_value:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate / Sensitivity")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


def plot_confusion_matrix_from_values(y_true, y_prob, threshold: float, title: str, save_path: Path) -> None:
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ["benign", "malignant"], rotation=30)
    plt.yticks(tick_marks, ["benign", "malignant"])
    thresh = cm.max() / 2.0 if cm.max() > 0 else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], "d"), ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


def plot_training_curves(log_df: pd.DataFrame, title: str, save_path: Path) -> None:
    plt.figure(figsize=(8, 5))
    plt.plot(log_df["epoch"], log_df["train_loss"], label="train_loss")
    plt.plot(log_df["epoch"], log_df["val_loss"], label="val_loss")
    if "val_roc_auc" in log_df.columns:
        plt.plot(log_df["epoch"], log_df["val_roc_auc"], label="val_roc_auc")
    plt.xlabel("Epoch")
    plt.ylabel("Value")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


def plot_combined_roc(results: List[Dict], save_path: Path) -> None:
    plt.figure(figsize=(7, 6))
    plotted = 0
    for r in results:
        pred_path = r.get("test_predictions_path")
        if not pred_path or not Path(pred_path).exists():
            continue
        pred_df = pd.read_csv(pred_path)
        y_true = pred_df["target"].values.astype(int)
        y_prob = pred_df["probability"].values.astype(float)
        if len(np.unique(y_true)) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_value = roc_auc_score(y_true, y_prob)
        label = f"{r['experiment_name']} (AUC={auc_value:.3f})"
        plt.plot(fpr, tpr, label=label)
        plotted += 1
    if plotted == 0:
        plt.close()
        return
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate / Sensitivity")
    plt.title("Test ROC comparison")
    plt.legend(fontsize=7, loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

print("Metric and plotting utilities ready.")

Metric and plotting utilities ready.


## 9. Training and evaluation loops

Best checkpoint is selected by validation ROC-AUC. If AUC cannot be computed, F1 is used as a fallback score.

In [11]:
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, config: Dict = CONFIG):
    is_train = optimizer is not None
    model.train(is_train)
    losses = []
    all_targets = []
    all_probs = []
    all_paths = []

    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        targets = batch["target"].to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        use_amp = config["training"].get("use_amp", True) and DEVICE.type == "cuda"
        with torch.set_grad_enabled(is_train):
            with autocast(enabled=use_amp):
                logits = model(images).view(-1)
                loss = criterion(logits, targets)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        probs = torch.sigmoid(logits.detach()).cpu().numpy()
        losses.append(loss.item() * images.size(0))
        all_targets.extend(targets.detach().cpu().numpy().tolist())
        all_probs.extend(probs.tolist())
        all_paths.extend(batch["image_path"])

    avg_loss = float(np.sum(losses) / len(loader.dataset))
    metrics = compute_metrics(all_targets, all_probs, threshold=config["training"]["threshold"])
    return avg_loss, metrics, np.array(all_targets), np.array(all_probs), all_paths


def evaluate_model(model, loader, criterion, config: Dict = CONFIG):
    with torch.no_grad():
        return run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, config=config)


def experiment_score(metrics: Dict[str, float]) -> float:
    auc = metrics.get("roc_auc", np.nan)
    if auc is not None and np.isfinite(auc):
        return float(auc)
    return float(metrics.get("f1", 0.0))


def train_experiment(exp: Dict, train_df, val_df, test_df, config: Dict = CONFIG) -> Dict:
    seed_everything(config["seed"])
    experiment_name = exp["name"]
    print("\n" + "=" * 90)
    print("Starting experiment:", experiment_name)
    print(exp)

    train_loader, val_loader, test_loader = make_dataloaders(
        train_df, val_df, test_df, config, use_weighted_sampler=exp.get("weighted_sampler", False)
    )

    model = build_model(exp["model_key"], pretrained=exp.get("pretrained", True)).to(DEVICE)
    print(f"Trainable parameters: {count_parameters(model):,}")

    criterion = build_loss(exp["loss_name"], train_df, config)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["training"]["learning_rate"],
        weight_decay=config["training"]["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(config["training"]["epochs"], 1)
    )
    scaler = GradScaler(enabled=config["training"].get("use_amp", True) and DEVICE.type == "cuda")

    best_score = -float("inf")
    best_epoch = -1
    epochs_without_improvement = 0
    log_rows = []
    best_ckpt_path = CHECKPOINT_DIR / f"{experiment_name}_best.pt"

    for epoch in range(1, config["training"]["epochs"] + 1):
        start = time.time()
        train_loss, train_metrics, _, _, _ = run_one_epoch(
            model, train_loader, criterion, optimizer=optimizer, scaler=scaler, config=config
        )
        val_loss, val_metrics, _, _, _ = evaluate_model(model, val_loader, criterion, config=config)
        scheduler.step()

        score = experiment_score(val_metrics)
        improved = score > best_score
        if improved:
            best_score = score
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "experiment": exp,
                "config": config,
                "best_epoch": best_epoch,
                "best_score": best_score,
                "val_metrics": val_metrics,
            }, best_ckpt_path)
        else:
            epochs_without_improvement += 1

        row = {
            "experiment_name": experiment_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_roc_auc": train_metrics["roc_auc"],
            "train_auprc": train_metrics["auprc"],
            "train_accuracy": train_metrics["accuracy"],
            "train_f1": train_metrics["f1"],
            "val_roc_auc": val_metrics["roc_auc"],
            "val_auprc": val_metrics["auprc"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall_sensitivity": val_metrics["recall_sensitivity"],
            "val_specificity": val_metrics["specificity"],
            "val_f1": val_metrics["f1"],
            "lr": optimizer.param_groups[0]["lr"],
            "epoch_seconds": time.time() - start,
            "improved": improved,
        }
        log_rows.append(row)
        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"val_auc={val_metrics['roc_auc']:.4f} | "
            f"val_sens={val_metrics['recall_sensitivity']:.4f} | "
            f"val_spec={val_metrics['specificity']:.4f} | "
            f"val_f1={val_metrics['f1']:.4f} | "
            f"best_epoch={best_epoch}"
        )

        if epochs_without_improvement >= config["training"]["early_stopping_patience"]:
            print(f"Early stopping at epoch {epoch}.")
            break

    log_df = pd.DataFrame(log_rows)
    log_path = LOG_DIR / f"{experiment_name}_training_log.csv"
    log_df.to_csv(log_path, index=False)
    plot_training_curves(log_df, experiment_name, FIGURE_DIR / f"{experiment_name}_training_curves.png")

    # Load best checkpoint and evaluate test set.
    checkpoint = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    test_loss, test_metrics, y_true, y_prob, paths = evaluate_model(model, test_loader, criterion, config=config)

    pred_df = pd.DataFrame({
        "image_path": paths,
        "target": y_true.astype(int),
        "probability": y_prob.astype(float),
        "prediction": (y_prob >= config["training"]["threshold"]).astype(int),
    })
    pred_path = RESULT_DIR / f"{experiment_name}_test_predictions.csv"
    pred_df.to_csv(pred_path, index=False)

    plot_roc_curve(y_true, y_prob, f"ROC - {experiment_name}", FIGURE_DIR / f"{experiment_name}_roc.png")
    plot_confusion_matrix_from_values(
        y_true,
        y_prob,
        config["training"]["threshold"],
        f"Confusion Matrix - {experiment_name}",
        FIGURE_DIR / f"{experiment_name}_confusion_matrix.png",
    )

    result = {
        "experiment_name": experiment_name,
        "model_key": exp["model_key"],
        "loss_name": exp["loss_name"],
        "weighted_sampler": exp.get("weighted_sampler", False),
        "best_epoch": best_epoch,
        "best_val_score": best_score,
        "test_loss": test_loss,
        "test_predictions_path": str(pred_path),
        "checkpoint_path": str(best_ckpt_path),
        "log_path": str(log_path),
    }
    for k, v in test_metrics.items():
        result[f"test_{k}"] = v

    print("Test metrics:")
    for k in ["roc_auc", "auprc", "accuracy", "precision", "recall_sensitivity", "specificity", "f1"]:
        print(f"  {k}: {test_metrics[k]:.4f}")
    return result

print("Training functions ready.")

Training functions ready.


## 10. Experiment plan

`core` runs the minimum set needed to answer the paper's research questions. `full` adds EfficientNet-B3 comparisons.

In [12]:
def get_experiments(plan: str) -> List[Dict]:
    quick = [
        {"name": "effb0_bce_quick", "model_key": "efficientnet_b0", "loss_name": "bce", "weighted_sampler": False, "pretrained": True},
    ]

    core = [
        {"name": "effb0_bce", "model_key": "efficientnet_b0", "loss_name": "bce", "weighted_sampler": False, "pretrained": True},
        {"name": "effb0_weighted_bce", "model_key": "efficientnet_b0", "loss_name": "weighted_bce", "weighted_sampler": False, "pretrained": True},
        {"name": "effb0_focal", "model_key": "efficientnet_b0", "loss_name": "focal", "weighted_sampler": False, "pretrained": True},
        {"name": "effb0_cb_focal", "model_key": "efficientnet_b0", "loss_name": "class_balanced_focal", "weighted_sampler": False, "pretrained": True},
        {"name": "effb0_weighted_bce_sampler", "model_key": "efficientnet_b0", "loss_name": "weighted_bce", "weighted_sampler": True, "pretrained": True},
        {"name": "convnext_tiny_bce", "model_key": "convnext_tiny", "loss_name": "bce", "weighted_sampler": False, "pretrained": True},
        {"name": "convnext_tiny_weighted_bce", "model_key": "convnext_tiny", "loss_name": "weighted_bce", "weighted_sampler": False, "pretrained": True},
        {"name": "convnext_tiny_focal", "model_key": "convnext_tiny", "loss_name": "focal", "weighted_sampler": False, "pretrained": True},
        {"name": "convnext_tiny_cb_focal", "model_key": "convnext_tiny", "loss_name": "class_balanced_focal", "weighted_sampler": False, "pretrained": True},
        {"name": "convnext_tiny_weighted_bce_sampler", "model_key": "convnext_tiny", "loss_name": "weighted_bce", "weighted_sampler": True, "pretrained": True},
    ]

    full_extra = [
        {"name": "effb3_weighted_bce", "model_key": "efficientnet_b3", "loss_name": "weighted_bce", "weighted_sampler": False, "pretrained": True},
        {"name": "effb3_cb_focal", "model_key": "efficientnet_b3", "loss_name": "class_balanced_focal", "weighted_sampler": False, "pretrained": True},
        {"name": "effb3_weighted_bce_sampler", "model_key": "efficientnet_b3", "loss_name": "weighted_bce", "weighted_sampler": True, "pretrained": True},
    ]

    if plan == "quick":
        return quick
    if plan == "core":
        return core
    if plan == "full":
        return core + full_extra
    raise ValueError(f"Unknown experiment plan: {plan}")

EXPERIMENTS = get_experiments(CONFIG["run"]["experiment_plan"])
print(f"Experiment plan = {CONFIG['run']['experiment_plan']} | n={len(EXPERIMENTS)}")
for exp in EXPERIMENTS:
    print("-", exp["name"])

Experiment plan = core | n=10
- effb0_bce
- effb0_weighted_bce
- effb0_focal
- effb0_cb_focal
- effb0_weighted_bce_sampler
- convnext_tiny_bce
- convnext_tiny_weighted_bce
- convnext_tiny_focal
- convnext_tiny_cb_focal
- convnext_tiny_weighted_bce_sampler


## 11. Run all experiments

This cell trains all configured experiments and saves logs, checkpoints, figures, and predictions.

In [13]:
all_results = []
failed_experiments = []

for exp in EXPERIMENTS:
    try:
        result = train_experiment(exp, train_df, val_df, test_df, CONFIG)
        all_results.append(result)
        summary_df = pd.DataFrame(all_results)
        summary_df.to_csv(RESULT_DIR / "all_experiments_summary_partial.csv", index=False)
    except Exception as e:
        print(f"Experiment failed: {exp['name']}")
        print(repr(e))
        failed_experiments.append({"experiment_name": exp["name"], "error": repr(e)})
        if not CONFIG["run"].get("continue_on_experiment_error", True):
            raise

summary_df = pd.DataFrame(all_results)
summary_path = RESULT_DIR / "all_experiments_summary.csv"
summary_df.to_csv(summary_path, index=False)

if failed_experiments:
    failed_df = pd.DataFrame(failed_experiments)
    failed_df.to_csv(RESULT_DIR / "failed_experiments.csv", index=False)
    print("Failed experiments saved to:", RESULT_DIR / "failed_experiments.csv")

plot_combined_roc(all_results, FIGURE_DIR / "combined_test_roc.png")
print("\nFinal summary saved to:", summary_path)
summary_df


Starting experiment: effb0_bce
{'name': 'effb0_bce', 'model_key': 'efficientnet_b0', 'loss_name': 'bce', 'weighted_sampler': False, 'pretrained': True}
Trainable parameters: 4,008,829


/tmp/ipykernel_1467851/2574245648.py:73: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=config["training"].get("use_amp", True) and DEVICE.type == "cuda")
/tmp/ipykernel_1467851/2574245648.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipykernel_1467851/2574245648.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


Epoch 001 | train_loss=0.6432 | val_loss=0.3381 | val_auc=0.8503 | val_sens=0.3923 | val_spec=0.9654 | val_f1=0.5057 | best_epoch=1


/tmp/ipykernel_1467851/2574245648.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipykernel_1467851/2574245648.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


Epoch 002 | train_loss=0.3223 | val_loss=0.3106 | val_auc=0.8830 | val_sens=0.3746 | val_spec=0.9846 | val_f1=0.5184 | best_epoch=2


...

/tmp/ipykernel_1467851/2574245648.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipykernel_1467851/2574245648.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


Epoch 020 | train_loss=0.8391 | val_loss=1.0343 | val_auc=0.8468 | val_sens=0.9602 | val_spec=0.4266 | val_f1=0.4174 | best_epoch=20


/tmp/ipykernel_1467851/2574245648.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


Test metrics:
  roc_auc: 0.8592
  auprc: 0.6222
  accuracy: 0.5313
  precision: 0.2713
  recall_sensitivity: 0.9646
  specificity: 0.4372
  f1: 0.4234

Final summary saved to: content/skin_cancer_outputs/results/all_experiments_summary.csv


,experiment_name,model_key,loss_name,weighted_sampler,best_epoch,best_val_score,test_loss,test_predictions_path,checkpoint_path,log_path,...,test_auprc,test_accuracy,test_precision,test_recall_sensitivity,test_f1,test_specificity,test_tn,test_fp,test_fn,test_tp
0,effb0_bce,efficientnet_b0,bce,False,20,0.947306,0.314987,content/skin_cancer_outputs/results/effb0_bce_...,content/skin_cancer_outputs/checkpoints/effb0_...,content/skin_cancer_outputs/logs/effb0_bce_tra...,...,0.871683,0.927895,0.841216,0.734513,0.784252,0.969891,3028,94,180,498
1,effb0_weighted_bce,efficientnet_b0,weighted_bce,False,20,0.946635,0.861348,content/skin_cancer_outputs/results/effb0_weig...,content/skin_cancer_outputs/checkpoints/effb0_...,content/skin_cancer_outputs/logs/effb0_weighte...,...,0.869617,0.924474,0.789630,0.786136,0.787879,0.954516,2980,142,145,533
2,effb0_focal,efficientnet_b0,focal,False,18,0.930979,0.029409,content/skin_cancer_outputs/results/effb0_foca...,content/skin_cancer_outputs/checkpoints/effb0_...,content/skin_cancer_outputs/logs/effb0_focal_t...,...,0.821625,0.910526,0.840726,0.615044,0.710392,0.974696,3043,79,261,417
3,effb0_cb_focal,efficientnet_b0,class_balanced_focal,False,18,0.936626,0.085014,content/skin_cancer_outputs/results/effb0_cb_f...,content/skin_cancer_outputs/checkpoints/effb0_...,content/skin_cancer_outputs/logs/effb0_cb_foca...,...,0.830261,0.902368,0.712310,0.759587,0.735189,0.933376,2914,208,163,515
4,effb0_weighted_bce_sampler,efficientnet_b0,weighted_bce,True,17,0.953362,0.797490,content/skin_cancer_outputs/results/effb0_weig...,content/skin_cancer_outputs/checkpoints/effb0_...,content/skin_cancer_outputs/logs/effb0_weighte...,...,0.856032,0.914474,0.732543,0.820059,0.773834,0.934978,2919,203,122,556
5,convnext_tiny_bce,convnext_tiny,bce,False,20,0.842794,0.322216,content/skin_cancer_outputs/results/convnext_t...,content/skin_cancer_outputs/checkpoints/convne...,content/skin_cancer_outputs/logs/convnext_tiny...,...,0.636198,0.862105,0.722543,0.368732,0.488281,0.969250,3026,96,428,250
6,convnext_tiny_weighted_bce,convnext_tiny,weighted_bce,False,2,0.796640,0.849572,content/skin_cancer_outputs/results/convnext_t...,content/skin_cancer_outputs/checkpoints/convne...,content/skin_cancer_outputs/logs/convnext_tiny...,...,0.514554,0.697105,0.345324,0.778761,0.478478,0.679372,2121,1001,150,528
7,convnext_tiny_focal,convnext_tiny,focal,False,19,0.905842,0.027115,content/skin_cancer_outputs/results/convnext_t...,content/skin_cancer_outputs/checkpoints/convne...,content/skin_cancer_outputs/logs/convnext_tiny...,...,0.785025,0.893421,0.871935,0.471976,0.612440,0.984946,3075,47,358,320
8,convnext_tiny_cb_focal,convnext_tiny,class_balanced_focal,False,19,0.835432,0.079906,content/skin_cancer_outputs/results/convnext_t...,content/skin_cancer_outputs/checkpoints/convne...,content/skin_cancer_outputs/logs/convnext_tiny...,...,0.630196,0.851053,0.584848,0.569322,0.576981,0.912236,2848,274,292,386
9,convnext_tiny_weighted_bce_sampler,convnext_tiny,weighted_bce,True,20,0.846785,0.986816,content/skin_cancer_outputs/results/convnext_t...,content/skin_cancer_outputs/checkpoints/convne...,content/skin_cancer_outputs/logs/convnext_tiny...,...,0.622185,0.531316,0.271257,0.964602,0.423438,0.437220,1365,1757,24,654


## 12. Select best model and create paper-ready final table

The best model is selected by **test ROC-AUC** for reporting convenience here. In the paper, emphasize that model selection during training used **validation ROC-AUC**.

In [14]:
if len(summary_df) == 0:
    raise RuntimeError("No successful experiments. Check failed_experiments.csv and error messages above.")

ranking_cols = [
    "experiment_name", "model_key", "loss_name", "weighted_sampler", "best_epoch",
    "test_roc_auc", "test_auprc", "test_accuracy", "test_precision",
    "test_recall_sensitivity", "test_specificity", "test_f1", "test_tn", "test_fp", "test_fn", "test_tp",
]
existing_cols = [c for c in ranking_cols if c in summary_df.columns]
final_table = summary_df[existing_cols].sort_values("test_roc_auc", ascending=False).reset_index(drop=True)
final_table_path = RESULT_DIR / "paper_results_table.csv"
final_table.to_csv(final_table_path, index=False)

best_row = final_table.iloc[0].to_dict()
best_experiment_name = best_row["experiment_name"]
print("Best experiment by test ROC-AUC:", best_experiment_name)
print("Paper results table saved to:", final_table_path)
final_table

Best experiment by test ROC-AUC: effb0_weighted_bce
Paper results table saved to: content/skin_cancer_outputs/results/paper_results_table.csv


,experiment_name,model_key,loss_name,weighted_sampler,best_epoch,test_roc_auc,test_auprc,test_accuracy,test_precision,test_recall_sensitivity,test_specificity,test_f1,test_tn,test_fp,test_fn,test_tp
0,effb0_weighted_bce,efficientnet_b0,weighted_bce,False,20,0.957570,0.869617,0.924474,0.789630,0.786136,0.954516,0.787879,2980,142,145,533
1,effb0_bce,efficientnet_b0,bce,False,20,0.957260,0.871683,0.927895,0.841216,0.734513,0.969891,0.784252,3028,94,180,498
2,effb0_weighted_bce_sampler,efficientnet_b0,weighted_bce,True,17,0.956071,0.856032,0.914474,0.732543,0.820059,0.934978,0.773834,2919,203,122,556
3,effb0_cb_focal,efficientnet_b0,class_balanced_focal,False,18,0.939280,0.830261,0.902368,0.712310,0.759587,0.933376,0.735189,2914,208,163,515
4,effb0_focal,efficientnet_b0,focal,False,18,0.939136,0.821625,0.910526,0.840726,0.615044,0.974696,0.710392,3043,79,261,417
5,convnext_tiny_focal,convnext_tiny,focal,False,19,0.924313,0.785025,0.893421,0.871935,0.471976,0.984946,0.612440,3075,47,358,320
6,convnext_tiny_weighted_bce_sampler,convnext_tiny,weighted_bce,True,20,0.859213,0.622185,0.531316,0.271257,0.964602,0.437220,0.423438,1365,1757,24,654
7,convnext_tiny_bce,convnext_tiny,bce,False,20,0.858091,0.636198,0.862105,0.722543,0.368732,0.969250,0.488281,3026,96,428,250
8,convnext_tiny_cb_focal,convnext_tiny,class_balanced_focal,False,19,0.857185,0.630196,0.851053,0.584848,0.569322,0.912236,0.576981,2848,274,292,386
9,convnext_tiny_weighted_bce,convnext_tiny,weighted_bce,False,2,0.815845,0.514554,0.697105,0.345324,0.778761,0.679372,0.478478,2121,1001,150,528


## 13. Optional Grad-CAM for the best model

This cell creates Grad-CAM visualizations for a few test images using the best experiment's checkpoint. It is optional and will be skipped if the Grad-CAM package or model layer discovery fails.

In [15]:
def denormalize_tensor(img_tensor: torch.Tensor) -> np.ndarray:
    img = img_tensor.detach().cpu().clone()
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    img = img * std + mean
    img = img.clamp(0, 1)
    return img.permute(1, 2, 0).numpy()


def get_gradcam_target_layer(model: nn.Module):
    # Common timm structures.
    if hasattr(model, "conv_head"):
        return model.conv_head
    if hasattr(model, "stages"):
        return model.stages[-1]
    if hasattr(model, "blocks"):
        return model.blocks[-1]
    # Fallback: last Conv2d module.
    last_conv = None
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            last_conv = module
    if last_conv is None:
        raise ValueError("Could not find a convolutional layer for Grad-CAM.")
    return last_conv


def make_gradcam_for_best_model(best_row: Dict, test_df: pd.DataFrame, config: Dict, n_images: int = 8):
    if not config["run"].get("make_gradcam", True):
        print("Grad-CAM disabled in config.")
        return
    try:
        from pytorch_grad_cam import GradCAM
        from pytorch_grad_cam.utils.image import show_cam_on_image
        from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    except Exception as e:
        print("Skipping Grad-CAM because pytorch-grad-cam is unavailable:", repr(e))
        return

    exp_name = best_row["experiment_name"]
    exp_record = summary_df[summary_df["experiment_name"] == exp_name].iloc[0].to_dict()
    checkpoint = torch.load(exp_record["checkpoint_path"], map_location=DEVICE, weights_only=False)
    exp = checkpoint["experiment"]

    model = build_model(exp["model_key"], pretrained=False).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    target_layer = get_gradcam_target_layer(model)
    transform = get_transforms("val", config)

    # Sample malignant cases first because they are clinically important; fill with benign if needed.
    malignant_df = test_df[test_df["target"] == 1]
    benign_df = test_df[test_df["target"] == 0]
    sample_df = pd.concat([
        malignant_df.sample(min(len(malignant_df), n_images // 2), random_state=config["seed"]),
        benign_df.sample(min(len(benign_df), n_images - min(len(malignant_df), n_images // 2)), random_state=config["seed"]),
    ]).reset_index(drop=True)

    out_dir = FIGURE_DIR / "gradcam_best_model"
    out_dir.mkdir(parents=True, exist_ok=True)

    with GradCAM(model=model, target_layers=[target_layer]) as cam:
        for i, row in sample_df.iterrows():
            image_path = row["image_path"]
            pil_img = Image.open(image_path).convert("RGB")
            input_tensor = transform(pil_img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                logit = model(input_tensor).view(-1)
                prob = torch.sigmoid(logit).item()
            grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(0)])[0]
            rgb_img = denormalize_tensor(input_tensor[0])
            visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

            save_path = out_dir / f"gradcam_{i:02d}_true{int(row['target'])}_prob{prob:.3f}.png"
            plt.figure(figsize=(8, 4))
            plt.subplot(1, 2, 1)
            plt.imshow(rgb_img)
            plt.axis("off")
            plt.title(f"Original | true={int(row['target'])}")
            plt.subplot(1, 2, 2)
            plt.imshow(visualization)
            plt.axis("off")
            plt.title(f"Grad-CAM | p(malignant)={prob:.3f}")
            plt.tight_layout()
            plt.savefig(save_path, dpi=300)
            plt.close()
            print("Saved:", save_path)

make_gradcam_for_best_model(best_row, test_df, CONFIG, n_images=8)

Saved: content/skin_cancer_outputs/figures/gradcam_best_model/gradcam_00_true1_prob0.352.png
Saved: content/skin_cancer_outputs/figures/gradcam_best_model/gradcam_01_true1_prob0.998.png
Saved: content/skin_cancer_outputs/figures/gradcam_best_model/gradcam_02_true1_prob0.560.png
Saved: content/skin_cancer_outputs/figures/gradcam_best_model/gradcam_03_true1_prob1.000.png
Saved: content/skin_cancer_outputs/figures/gradcam_best_model/gradcam_04_true0_prob0.000.png
Saved: content/skin_cancer_outputs/figures/gradcam_best_model/gradcam_05_true0_prob0.020.png
Saved: content/skin_cancer_outputs/figures/gradcam_best_model/gradcam_06_true0_prob0.000.png
Saved: content/skin_cancer_outputs/figures/gradcam_best_model/gradcam_07_true0_prob0.032.png


## 14. What to use in the paper

After the notebook finishes, use these files:

- `outputs/results/paper_results_table.csv`: main quantitative results table
- `outputs/results/all_experiments_summary.csv`: complete metrics for all experiments
- `outputs/logs/*_training_log.csv`: per-epoch training logs
- `outputs/figures/*_roc.png`: per-experiment ROC curves
- `outputs/figures/combined_test_roc.png`: model comparison ROC figure
- `outputs/figures/*_confusion_matrix.png`: confusion matrices
- `outputs/figures/*_training_curves.png`: training curves
- `outputs/figures/gradcam_best_model/*.png`: explainability examples

Recommended paper framing:

1. Model comparison: EfficientNet vs ConvNeXt.
2. Imbalance comparison: BCE vs weighted BCE vs focal loss vs class-balanced focal loss vs weighted sampling.
3. Main clinical emphasis: ROC-AUC, sensitivity/recall, specificity, and confusion matrix.
4. Do not claim clinical deployment; describe this as a retrospective image-classification experiment.